In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

loader = PyMuPDFLoader('../data/medical_report.pdf')
docs = loader.load()
len(docs)

splitter_docs = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter_docs.split_documents(docs)
len(splitted_docs)

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-SMALL')
vectorstore = InMemoryVectorStore(
    documents = splitted_docs,
    embedding=embeddings
)

same_records = vectorstore.similarity_search('patient name')
len(same_records) #Default: 4
print(same_records[0].page_content)

#AGENT: Tools, LLM, SystemPrompt
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retreive the relevant data of PDF Document
        Which have details about medical reports
    """

    print('tool called', query)

    docs = vectorstore.similarity_search(query, k=4)
    context = ''
    for doc in docs:
        context += docs
    print(len(docs), docs[0].page_content)
    return context

retriever_tool.invoke('Patient Name')

llm = ChatOpenAI(model='gpt-5')
system_prompt = '''
    You are a helpful assistent that answers question usning provided context
'''

agent = create_agent(
    model = llm,
    tools = [retriever_tool],
    system_prompt=system_prompt
)

query = 'What is the name of patient and name of doctors?'
response = agent.invoke({'messages':[{'role': 'user', 'content':query}]})
result = response['messages'][-1].content
print(result)